In [1]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [2]:
import torch
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import precision_recall_curve, auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import optuna
from rdkit.Chem import Descriptors, AllChem
from tqdm import tqdm  # 导入tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold





In [3]:
# 函数：将SMILES转换为分子描述符和指纹
def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # 提取描述符
    descriptors = [
        Descriptors.MolWt(mol),  # 分子量
        Descriptors.MolLogP(mol),  # LogP
        Descriptors.NumHDonors(mol),  # 氢键供体数量
        Descriptors.NumHAcceptors(mol)  # 氢键受体数量
    ]
    # 生成Morgan指纹
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fingerprint_array = np.zeros((2048,))
    Chem.DataStructs.ConvertToNumpyArray(fingerprint, fingerprint_array)
    # 合并描述符和指纹
    features = np.concatenate([descriptors, fingerprint_array])
    return features


In [4]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import optuna
import numpy as np

def train_evaluate_regression_model_with_optuna(model_name, model_class, param_func, X, y, groups):
    def objective(trial):
        params = param_func(trial)
        model = model_class(**params)

        gkf = GroupKFold(n_splits=10)
        maes = []

        for train_idx, val_idx in tqdm(gkf.split(X, y, groups=groups), total=10, desc=f"Training {model_name}"):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            # ✅ 计算 MAE
            mae = mean_absolute_error(y_val, y_pred)
            maes.append(mae)

        return np.mean(maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print(f'Best parameters for {model_name}: {study.best_params}')
    print(f'Best mean MAE: {study.best_value:.4f}')

In [5]:
# 数据预处理
df = pd.read_excel('../invertebrates_EC10_unique.xlsx')
labels = df['mgperL'].values
smiles_list = df['SMILES_Canonical_RDKit'].tolist()
endpoints_a = df['endpoint']
Duration_Values_a = df['Duration_Value'].values
effects_a = df['effect']


In [6]:

features = []
new_labels = []
new_smiles_list = []
endpoints = []
Duration_Values = []
effects =[]


for smiles, label,a,b,c in zip(smiles_list, labels,Duration_Values_a,effects_a,endpoints_a):
    feature = smiles_to_features(smiles)
    if feature is not None:
        features.append(feature)
        new_labels.append(label)
        new_smiles_list.append(smiles)
        Duration_Values.append(a)
        effects.append(b)
        endpoints.append(c)

X = np.array(features)
y = np.array(new_labels)
groups = new_smiles_list  # 可直接用于 GroupKFold




In [7]:
def encode_column(zz):
    zz_series = pd.Series(zz)  # 转换为 Series
    unique_values = zz_series.unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(zz_series.values.reshape(-1, 1))
    else:
        return None  # 只有一种类别时忽略

Duration_Values =pd.Series(Duration_Values)


# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(effects)
endpoint_encoded = encode_column(endpoints)
#species_encoded = encode_column(df, 'species_group')

# # 将需要的列拼接成输入 X
X = np.hstack((X, Duration_Values.values.reshape(-1, 1)))

# # 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded]:
     if encoded_feature is not None:
         X = np.hstack((X, encoded_feature))



y=np.log1p(y)

In [8]:
def xgb_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),   # L1 正则
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)  # L2 正则
    }
from xgboost import XGBRegressor

train_evaluate_regression_model_with_optuna(
    "XGBoost",
    XGBRegressor,
    xgb_param_func,
    X, y, groups
)

[I 2025-05-15 16:09:59,115] A new study created in memory with name: no-name-fdb50bfd-f352-467b-b404-d409499be8bd
Training XGBoost: 100%|██████████| 10/10 [02:04<00:00, 12.46s/it]
[I 2025-05-15 16:12:03,757] Trial 0 finished with value: 0.856262307210876 and parameters: {'n_estimators': 581, 'max_depth': 15, 'learning_rate': 0.12593637688021947, 'subsample': 0.9994352650347863, 'colsample_bytree': 0.8879461961579502, 'reg_alpha': 0.9559193612488608, 'reg_lambda': 0.09039298703595178}. Best is trial 0 with value: 0.856262307210876.
Training XGBoost: 100%|██████████| 10/10 [01:28<00:00,  8.87s/it]
[I 2025-05-15 16:13:32,550] Trial 1 finished with value: 0.8757297947643734 and parameters: {'n_estimators': 322, 'max_depth': 11, 'learning_rate': 0.020813330272992356, 'subsample': 0.8039968657972181, 'colsample_bytree': 0.7784735703586152, 'reg_alpha': 0.21980675173943598, 'reg_lambda': 0.8343849388633017}. Best is trial 0 with value: 0.856262307210876.
Training XGBoost: 100%|██████████| 10/

Best parameters for XGBoost: {'n_estimators': 495, 'max_depth': 16, 'learning_rate': 0.03842576072234601, 'subsample': 0.9718243029318829, 'colsample_bytree': 0.8852872221023105, 'reg_alpha': 0.6081099042998649, 'reg_lambda': 0.12425583595665979}
Best mean MAE: 0.8457


In [9]:
from lightgbm import LGBMRegressor

def lgbm_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'verbose': -1
    }

print("Training LightGBM (Poisson)...")
train_evaluate_regression_model_with_optuna(
    "LightGBM",
    lambda **params: LGBMRegressor(objective="poisson", **params),  # ✅ 加入 Poisson 目标
    lgbm_param_func,
    X, y, groups
)

[I 2025-05-15 17:22:07,265] A new study created in memory with name: no-name-09a0fb1b-1ee9-49bd-8688-28a459b23f8d


Training LightGBM (Poisson)...


Training LightGBM: 100%|██████████| 10/10 [00:34<00:00,  3.47s/it]
[I 2025-05-15 17:22:41,998] Trial 0 finished with value: 0.8358937942647324 and parameters: {'n_estimators': 472, 'max_depth': 17, 'num_leaves': 144, 'learning_rate': 0.020820450583670644, 'feature_fraction': 0.820761613996074, 'bagging_fraction': 0.8176175415426429, 'bagging_freq': 3, 'reg_alpha': 0.025146924678517535, 'reg_lambda': 0.6481183157840131}. Best is trial 0 with value: 0.8358937942647324.
Training LightGBM: 100%|██████████| 10/10 [00:06<00:00,  1.54it/s]
[I 2025-05-15 17:22:48,518] Trial 1 finished with value: 0.8695458675518035 and parameters: {'n_estimators': 196, 'max_depth': 8, 'num_leaves': 92, 'learning_rate': 0.0665542437583174, 'feature_fraction': 0.7240957200143276, 'bagging_fraction': 0.7098041533253925, 'bagging_freq': 1, 'reg_alpha': 0.19149823154047707, 'reg_lambda': 0.08967439744242856}. Best is trial 0 with value: 0.8358937942647324.
Training LightGBM: 100%|██████████| 10/10 [00:16<00:00,  1.

Best parameters for LightGBM: {'n_estimators': 461, 'max_depth': 13, 'num_leaves': 221, 'learning_rate': 0.15946292009756194, 'feature_fraction': 0.7444819519285434, 'bagging_fraction': 0.9629117869301326, 'bagging_freq': 1, 'reg_alpha': 0.45234683161915346, 'reg_lambda': 0.6065449711479824}
Best mean MAE: 0.7938


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import optuna
import numpy as np


class DNNWithSoftplus(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_fn = {
            'relu': nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_fn]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softplus(self.net(x)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_dnn_with_optuna_pytorch(X, y, groups, device=device):
    def dnn_param_func(trial):
        return {
            'hidden_layer_sizes': trial.suggest_categorical(
                'hidden_layer_sizes', [(50,), (100,), (150,), (100, 50), (150, 100, 50)]
            ),
            'activation': trial.suggest_categorical('activation', ['relu', 'logistic', 'tanh']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
            'learning_rate': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'optimizer': trial.suggest_categorical('solver', ['adam', 'sgd'])
        }

    def objective(trial):
        params = dnn_param_func(trial)
        model = DNNWithSoftplus(
            input_dim=X.shape[1],
            hidden_sizes=params['hidden_layer_sizes'],
            activation=params['activation']
        ).to(device)

        optimizer = {
            'adam': torch.optim.Adam,
            'sgd': torch.optim.SGD
        }[params['optimizer']](model.parameters(), lr=params['learning_rate'], weight_decay=params['alpha'])

        loss_fn = nn.MSELoss()
        gkf = GroupKFold(n_splits=10)
        fold_maes = []

        for train_idx, val_idx in gkf.split(X, y, groups=groups):
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
            train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

            model.train()
            for epoch in range(100):
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    loss.backward()
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                val_preds = model(torch.tensor(X_val).float().to(device)).cpu().numpy()
                mae = mean_absolute_error(y_val, val_preds)
                fold_maes.append(mae)

        return np.mean(fold_maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)
    print("\n✅ Best Parameters Found:")
    print(study.best_params)
    print(f"Mean MAE = {study.best_value:.4f}")
    return study.best_params


best_dnn_params = train_dnn_with_optuna_pytorch(X, y, groups)

[I 2025-05-15 17:42:14,081] A new study created in memory with name: no-name-4cf311e4-5af9-4ac1-bef6-8b96c4aec15c
[I 2025-05-15 17:43:35,798] Trial 0 finished with value: 1.1180738589040533 and parameters: {'hidden_layer_sizes': (50,), 'activation': 'logistic', 'alpha': 0.0004332684300708851, 'learning_rate_init': 0.00015405209752252983, 'solver': 'sgd'}. Best is trial 0 with value: 1.1180738589040533.
[I 2025-05-15 17:45:03,775] Trial 1 finished with value: 0.6149237804208075 and parameters: {'hidden_layer_sizes': (150,), 'activation': 'logistic', 'alpha': 0.0003168814618562341, 'learning_rate_init': 0.00014988318239545794, 'solver': 'adam'}. Best is trial 1 with value: 0.6149237804208075.
[I 2025-05-15 17:46:32,511] Trial 2 finished with value: 0.49415330739808255 and parameters: {'hidden_layer_sizes': (100, 50), 'activation': 'relu', 'alpha': 0.00014987168330749367, 'learning_rate_init': 0.0013278465540447408, 'solver': 'sgd'}. Best is trial 2 with value: 0.49415330739808255.
[I 202


✅ Best Parameters Found:
{'hidden_layer_sizes': (150, 100, 50), 'activation': 'relu', 'alpha': 2.4216263420142954e-05, 'learning_rate_init': 0.00025492088579764395, 'solver': 'adam'}
Mean MAE = 0.3482
